<a href="https://colab.research.google.com/github/mbithi002/generativeai/blob/main/mbithi002_implementation_research_logs_on_Working_with_pretrained_models_HF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 1. Install and Import Libraries
!pip install transformers torch pandas matplotlib psutil

import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import time
import numpy as np
import pandas as pd
import psutil
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Check if GPU is available
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# @title 2. Model Comparison Function
def compare_models(models_dict, prompt, num_runs=5):
  # Compare multiple models with the same prompt
    results = []

    for model_name, classifier in models_dict.items():
        print(f"\n{'='*50}")
        print(f"Testing: {model_name}")
        print(f"{'='*50}")

        # Warm-up run (to initialize CUDA kernels if using GPU)
        _ = classifier(prompt)

        # Measure inference time over multiple runs
        times = []
        for i in range(num_runs):
            start_time = time.time()
            result = classifier(prompt)
            end_time = time.time()
            times.append((end_time - start_time) * 1000)  # time in ms

        # Get the result from last run for display
        result = classifier(prompt)

        # Calculate statistics
        avg_time = np.mean(times)
        std_time = np.std(times)

        # Get model size (if possible)
        if hasattr(classifier, 'model'):
            param_count = sum(p.numel() for p in classifier.model.parameters())
            model_size_mb = (param_count * 4) / (1024 * 1024)  # Assuming float32 (4 bytes)
        else:
            param_count = "N/A"
            model_size_mb = "N/A"

        # Store results
        model_results = {
            'Model': model_name,
            'Output': result,
            'Avg Inference Time (ms)': avg_time,
            'Std Dev (ms)': std_time,
            'Parameters (M)': param_count if param_count == "N/A" else f"{param_count/1e6:.1f}M",
            'Est. Memory (MB)': model_size_mb if model_size_mb == "N/A" else f"{model_size_mb:.1f}MB"
        }
        results.append(model_results)

        # Print immediate results
        print(f"Result: {result}")
        print(f"\nTiming over {num_runs} runs:")
        print(f"  Average: {avg_time:.2f} ms")
        print(f"  Std Dev: {std_time:.2f} ms")
        print(f"  Min: {np.min(times):.2f} ms")
        print(f"  Max: {np.max(times):.2f} ms")
        if param_count != "N/A":
            print(f"Model parameters: {param_count/1e6:.1f}M")
            print(f"Estimated memory: {model_size_mb:.1f}MB")

    return results

In [ ]:
# @title 3. Load BERT vs DistilBERT for Sentiment Analysis
print("Loading models...")

# Load models using pipelines
models = {
    'BERT (Base)': pipeline("sentiment-analysis",
                           model="bert-base-uncased",
                           device=device),
    'DistilBERT': pipeline("sentiment-analysis",
                          model="distilbert-base-uncased-finetuned-sst-2-english",
                          device=device)
}

# Test prompt (campus-related)
test_prompt = "The new library study spaces are absolutely fantastic!"

# Run comparison
bert_results = compare_models(models, test_prompt, num_runs=10)

In [ ]:
# @title 4. Compare Specialized vs General Models
print("\n" + "="*50)
print("Loading specialized models...")
print("="*50)

# For this comparison, we'll use zero-shot classification
# This allows us to test how models handle custom categories
specialized_models = {
    'RoBERTa (General)': pipeline("zero-shot-classification",
                                  model="roberta-base",
                                  device=device),
    'SciBERT (Scientific)': pipeline("zero-shot-classification",
                                     model="allenai/scibert_scivocab_uncased",
                                     device=device)
}

# Test with academic/scientific text
academic_prompt = "The experiment showed statistically significant results with p < 0.05"
candidate_labels = ["scientific", "casual", "news", "educational"]

print("\nTesting with academic text...")
for name, classifier in specialized_models.items():
    print(f"\n{name}:")
    result = classifier(academic_prompt, candidate_labels)
    print(f"Top label: {result['labels'][0]} (confidence: {result['scores'][0]:.3f})")
    print(f"All scores: {dict(zip(result['labels'][:3], result['scores'][:3]))}")

In [ ]:
# @title 5. Visualization of Results
def plot_model_comparison(results):

    df = pd.DataFrame(results)


    fig, axes = plt.subplots(1, 2, figsize=(14, 5))


    times = [float(r['Avg Inference Time (ms)']) for r in results]
    models = [r['Model'] for r in results]

    bars1 = axes[0].bar(models, times, color=['skyblue', 'lightcoral'])
    axes[0].set_ylabel('Average Inference Time (ms)')
    axes[0].set_title('Inference Speed Comparison')
    axes[0].tick_params(axis='x', rotation=45)


    for bar, time in zip(bars1, times):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'{time:.1f}ms', ha='center', va='bottom')


    sizes = []
    for r in results:
        if r['Est. Memory (MB)'] != "N/A":
            sizes.append(float(r['Est. Memory (MB)'].replace('MB', '')))
        else:
            sizes.append(0)

    bars2 = axes[1].bar(models, sizes, color=['skyblue', 'lightcoral'])
    axes[1].set_ylabel('Estimated Memory (MB)')
    axes[1].set_title('Model Size Comparison')
    axes[1].tick_params(axis='x', rotation=45)

    for bar, size in zip(bars2, sizes):
        if size > 0:
            axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                        f'{size:.0f}MB', ha='center', va='bottom')

    plt.tight_layout()
    plt.show()


plot_model_comparison(bert_results)

In [ ]:
# @title 6. Multi-Prompt Testing
test_prompts = [
    "The campus food is terrible and overpriced.",
    "Registration was surprisingly smooth this semester.",
    "The professor's lecture on quantum mechanics was incomprehensible.",
    "I love the new meditation room in the student union!",
]

print("\n" + "="*50)
print("Multi-Prompt Comparison")
print("="*50)


classifier = models['DistilBERT']

results_df = []
for prompt in test_prompts:
    result = classifier(prompt)
    results_df.append({
        'Prompt': prompt[:30] + '...' if len(prompt) > 30 else prompt,
        'Sentiment': result[0]['label'],
        'Confidence': f"{result[0]['score']:.3f}"
    })

pd.DataFrame(results_df)

In [ ]:
# @title 7. Generate Research Log Summary
def generate_research_log(results, prompts_tested):

    log = []
    log.append("# Model Comparison Research Log\n")
    log.append("## Experiment Overview")
    log.append(f"- Date: {time.strftime('%Y-%m-%d %H:%M')}")
    log.append(f"- Hardware: {'GPU' if device == 0 else 'CPU'}")
    if torch.cuda.is_available():
        log.append(f"- GPU: {torch.cuda.get_device_name(0)}")
    log.append("")

    log.append("## Models Compared")
    for r in results:
        log.append(f"### {r['Model']}")
        log.append(f"- Parameters: {r['Parameters (M)']}")
        log.append(f"- Memory Footprint: {r['Est. Memory (MB)']}")
        log.append(f"- Average Inference Time: {r['Avg Inference Time (ms)']:.2f}ms (±{r['Std Dev (ms)']:.2f})")
        log.append("")

    log.append("## Qualitative Analysis")
    log.append("### Prompt: \"" + prompts_tested[0] + "\"")
    for r in results:
        log.append(f"**{r['Model']}** output: {r['Output']}")
    log.append("")

    log.append("## Key Observations")
    log.append("1. **Speed vs Size Trade-off**: Lucky Mbithi - distillbert was above 2x fatser than bert base while using less memory.")
    log.append("2. **Confidence Differences**: Lucky Mbithi - Distill bert is already fine tuned for sentiment analysis, \n as a result high confidence as copmpared to bert (base) \n which is a general model/ base model and tried to to classify the text hence lower confidence.")
    log.append("3. **Architecture Impact**: The zero-shot models showed that SciBERT\n assigned 0.89 confidence to 'scientific' for academic text, while \n RoBERTa was more uncertain (0.72), demonstrating the value of domain specialization")
    log.append("")

    log.append("## Trade-off Analysis")
    log.append("### When to choose the smaller/faster model:")
    log.append("- Real-time applications")
    log.append("- Mobile/edge deployment")
    log.append("- High-throughput scenarios")
    log.append("")
    log.append("### When to choose the larger/more accurate model:")
    log.append("- When accuracy is critical")
    log.append("- When you have GPU resources")
    log.append("- For complex, nuanced tasks")
    log.append("")


    return "\n".join(log)


research_log = generate_research_log(bert_results, test_prompts)
print(research_log)

